# Phase 1: MVP, a coding agent that can fix a bug safely

Phase 0's agent can only **read**. It can explain a bug but not fix it. This phase gives it the tools to explore, edit and run code, with safety checks before anything risky.

This notebook is the lab: each step's plan, evidence, code and observations live here. [phase_1_README.md](phase_1_README.md) is written once the phase is finished (see [WORKFLOW.md](../WORKFLOW.md)).

## Goal

**Done when:** given a small project with a failing test, the agent

1. finds the bug,
2. fixes it,
3. runs the tests to confirm the fix,
4. and asks before anything risky.

## Plan

One section per roadmap step, built strictly in this order. Each step starts from what already exists and changes only what it needs.

| # | Section | What it adds | Why at this point |
|---|---|---|---|
| 1 | Tool registry | One place that lists the tools; the prompt, parser and loop read from it | Set up before adding tools, so each new tool is a function, not a prompt rewrite |
| 2 | List and find files | Tools to see what's in a project | The agent has to explore a project before it can change it |
| 3 | Folder boundary | File tools refuse paths outside the project | Before any tool can write |
| 4 | Write and edit files | Tools that change files | The moment the agent can actually change code |
| 5 | Approval prompt | Ask the user before risky actions | Needed before shell access |
| 6 | Run shell commands | A tool that runs commands | Lets the agent run code, tests and git |
| 7 | Verify loop | Run the tests, fix what fails, repeat | What makes it a *coding* agent |

After section 7: an end-to-end demo against the goal above. Then this notebook's code is refactored into `agent_v1/` (see [WORKFLOW.md](../WORKFLOW.md)).

## How this notebook works

- It imports **only** Phase 0's frozen package, `agent_v0`. Nothing comes from `agent_v1`, which doesn't exist yet.
- Everything new is built here. To change a Phase 0 function, we look at the imported version, then write a new one **with the same name** in this notebook (*shadowing*). We never modify `agent_v0` itself.
- **Every step starts with a game plan**: the problem, how real agents solve it, the proposed sub-steps, decisions and acceptance criteria. We agree on it before any code, then build the sub-steps one at a time.

## 0. Setup: import Phase 0

`phase-0-foundation` has a hyphen in its name, so Python can't import it as a package. Instead we add the folder to Python's search path, then import `agent_v0` from inside it.

In [2]:
import sys

sys.path.insert(0, "../phase-0-foundation")

from agent_v0 import SYSTEM_PROMPT, ask, call_llm, new_conversation, parse_tool_call, read_file

A quick check that it's the Phase 0 agent we know: its prompt still describes one hand-written tool.

In [3]:
print(SYSTEM_PROMPT)

You are a helpful coding assistant.

You have ONE tool:
  read_file(filename) - returns the contents of a text file.

To use it, reply with exactly one line and nothing else:
tool: read_file({"filename": "path/to/file"})

After you receive a tool_result(...) message, use it to continue the task.
If you don't need the tool, just answer normally.


## 1. Tool registry: game plan

*Agreed 2026-10-31. The sub-steps below are built one at a time, in order.*

### What we're doing and why

Phase 0 has one tool, `read_file`, and its name is written by hand in **three places** in `agent_v0/loop.py`:

| Place | Lines | What's hard-coded |
|---|---|---|
| The system prompt | [`loop.py:10-14`](../phase-0-foundation/agent_v0/loop.py) | "You have ONE tool: read_file(filename) ..." |
| The parser | [`loop.py:22`](../phase-0-foundation/agent_v0/loop.py) | `prefix = "tool: read_file("` |
| The loop | [`loop.py:51-52`](../phase-0-foundation/agent_v0/loop.py) | `result = read_file(args.get("filename", ""))` |

So adding a second tool would mean editing all three. The roadmap's goal for this step ([`ROADMAP.md:181`](../ROADMAP.md)): *"Describe tools automatically from their code; add a tool without touching the loop."* It comes first in Phase 1 because steps 2, 4 and 6 all add tools ([`ROADMAP.md:70`](../ROADMAP.md)).

### Evidence: how real agents do it

The notebook is how *we* do it. This table is how everyone else does: small-scale projects first, then the five big open-source industry agents (Gemini CLI, Codex CLI, opencode, OpenHands, Aider). Source code read on 2026-10-31; all links are pinned to the commit read. **This is a snapshot:** these repos change daily, so the table describes them as of that date and isn't updated afterwards.

| Tier | Agent | How it registers tools | Source |
|---|---|---|---|
| Small scale | **learn-claude-code** (lesson s02) | Two separate pieces: a hand-written list of tool descriptions (`TOOLS`) and a dictionary from name to function (`TOOL_HANDLERS`). The loop calls `TOOL_HANDLERS[name](**args)`; an unknown name returns `"Unknown: <name>"`. Its own summary: *"the loop stays the same; only tool registration and dispatch grow."* | [`s02_tool_use/code.py:138-156, 180-181`](https://github.com/shareAI-lab/learn-claude-code/blob/7226f555d48900076eba126acf37c9ed5f002b5d/s02_tool_use/code.py#L138-L181) |
| Small scale | **ghuntley/how-to-build-a-coding-agent** (Go) | One bundle per tool, `ToolDefinition`: name, description, input schema and function together. The description is hand-written; the input schema is **generated** from a type. The loop searches the tool list by name; an unknown name returns "tool not found". | [`read.go:253-271, 147-170`](https://github.com/ghuntley/how-to-build-a-coding-agent/blob/fa3dd49fdddf45e92b35c66746e6bb6aac6ac27e/read.go#L253-L271) |
| Small scale | **mini-swe-agent** (text mode) | **No registry.** One action type, a shell command, pulled out of the reply with a regex. A badly formatted reply is sent back to the model as an error message. | [`actions_text.py:15-40`](https://github.com/SWE-agent/mini-swe-agent/blob/04d809ceab9df28f9adaed044884180159172930/src/minisweagent/models/utils/actions_text.py#L15-L40) |
| Industry | **Gemini CLI** (Google, TypeScript, ~109k stars) | A `ToolRegistry` class holding a map from name to tool object; `registerTool` overwrites a duplicate name with a warning. Tools come from three sources: built in, **discovered** by running a project command, and MCP servers. The list sent to the model is rebuilt from each tool's own schema, **minus tools excluded in config**; in Plan Mode it **rewrites the write and edit tools' descriptions** to "only for plans". An unknown tool returns `Tool "x" not found.` plus **"did you mean" suggestions**: the 3 closest names by edit distance. Descriptions are hand-written in separate files per model family, and **the same tool gets different descriptions for different models**: for `read_file`, the Gemini 3 set says the model "MUST use 'start_line' and 'end_line' for targeted, surgical reads"; the default set doesn't. | [`tool-registry.ts:247-294`](https://github.com/google-gemini/gemini-cli/blob/6814e00d6b148c3a1d57244d5d7040f7c541009d/packages/core/src/tools/tool-registry.ts#L247-L294), [`:370-530`](https://github.com/google-gemini/gemini-cli/blob/6814e00d6b148c3a1d57244d5d7040f7c541009d/packages/core/src/tools/tool-registry.ts#L370-L530), [`:663-718`](https://github.com/google-gemini/gemini-cli/blob/6814e00d6b148c3a1d57244d5d7040f7c541009d/packages/core/src/tools/tool-registry.ts#L663-L718); [`scheduler.ts:374-419`](https://github.com/google-gemini/gemini-cli/blob/6814e00d6b148c3a1d57244d5d7040f7c541009d/packages/core/src/scheduler/scheduler.ts#L374-L419); [`tool-utils.ts:65`](https://github.com/google-gemini/gemini-cli/blob/6814e00d6b148c3a1d57244d5d7040f7c541009d/packages/core/src/utils/tool-utils.ts#L65); [`default-legacy.ts:84-86`](https://github.com/google-gemini/gemini-cli/blob/6814e00d6b148c3a1d57244d5d7040f7c541009d/packages/core/src/tools/definitions/model-family-sets/default-legacy.ts#L84-L86), [`gemini-3.ts:93-95`](https://github.com/google-gemini/gemini-cli/blob/6814e00d6b148c3a1d57244d5d7040f7c541009d/packages/core/src/tools/definitions/model-family-sets/gemini-3.ts#L93-L95), [`resolver.ts:17-34`](https://github.com/google-gemini/gemini-cli/blob/6814e00d6b148c3a1d57244d5d7040f7c541009d/packages/core/src/tools/definitions/resolver.ts#L17-L34) |
| Industry | **Codex CLI** (OpenAI, Rust, ~132k stars) | A `ToolRegistry` holding an ordered map from name to tool. **Two ways in**: `register_trusted` for built-in tools and `register_external` for outside tools, which must first pass a **tool policy**. Each built-in tool's description and parameters are **hand-written in code**, and vary by situation (the shell tool adds Windows guidance; approval parameters are added to its schema). An unknown tool returns `"unsupported call: <name>"` to the model, and the failed call is **logged to telemetry** (OpenTelemetry). | [`registry.rs:428-433, 516-580`](https://github.com/openai/codex/blob/67f0994b532b8740f14b5b738af7b7b2a915b1d6/codex-rs/core/src/tools/registry.rs#L428-L580), [`:766-786, 1148-1151`](https://github.com/openai/codex/blob/67f0994b532b8740f14b5b738af7b7b2a915b1d6/codex-rs/core/src/tools/registry.rs#L766-L786); [`shell_spec.rs:14-108`](https://github.com/openai/codex/blob/67f0994b532b8740f14b5b738af7b7b2a915b1d6/codex-rs/core/src/tools/handlers/shell_spec.rs#L14-L108) |
| Industry | **opencode** (independent, TypeScript, ~224k stars) | A registry keyed by tool name, with an editor API to add, update and remove tools; each tool is **validated when it's registered** (bad names or schemas become registration errors). Before each model call it takes a **snapshot** of the tools, dropping any that the agent's **permission rules** deny, so the model never sees them. Each parameter carries its own hand-written description. An unknown tool returns *"No tool named "x" is currently available. Please use a tool from the available tool list."* It also has a **Code Mode**: instead of calling tools one at a time, the model writes a small program that calls them. | [`tool.ts:22-42`](https://github.com/anomalyco/opencode/blob/a5518690fc5cfd57ea2abfb636ef03a9ce67fc51/packages/core/src/tool.ts#L22-L42), [`:265-290`](https://github.com/anomalyco/opencode/blob/a5518690fc5cfd57ea2abfb636ef03a9ce67fc51/packages/core/src/tool.ts#L265-L290), [`:328-335`](https://github.com/anomalyco/opencode/blob/a5518690fc5cfd57ea2abfb636ef03a9ce67fc51/packages/core/src/tool.ts#L328-L335), [`:381-414`](https://github.com/anomalyco/opencode/blob/a5518690fc5cfd57ea2abfb636ef03a9ce67fc51/packages/core/src/tool.ts#L381-L414); [`tool/plugin/read.ts:17-43`](https://github.com/anomalyco/opencode/blob/a5518690fc5cfd57ea2abfb636ef03a9ce67fc51/packages/core/src/tool/plugin/read.ts#L17-L43) |
| Industry | **OpenHands** (All Hands AI, Python SDK; main repo ~92k stars) | A **Python dictionary** from name to factory: `register_tool(name, factory)` takes a ready tool or a class that builds tools per conversation; a duplicate name logs a warning and overwrites. Each tool's arguments are a typed (Pydantic) class whose fields carry hand-written descriptions, and the terminal tool has **separate Unix and Windows descriptions**. Ships **model-specific tool sets** (e.g. Gemini-style file tools). An unknown tool returns *"Tool 'x' not found. Available: ..."*; **wrong arguments return a validation error to the model**, and it even tries to detect and repair some malformed calls. | [`tool/registry.py:34, 130-175`](https://github.com/OpenHands/software-agent-sdk/blob/98de8bf3aca7e59eedebd7a97c423986ea1c4c05/openhands-sdk/openhands/sdk/tool/registry.py#L130-L175); [`agent/agent.py:1414-1417, 1481`](https://github.com/OpenHands/software-agent-sdk/blob/98de8bf3aca7e59eedebd7a97c423986ea1c4c05/openhands-sdk/openhands/sdk/agent/agent.py#L1414-L1481); [`terminal/definition.py:56-102`](https://github.com/OpenHands/software-agent-sdk/blob/98de8bf3aca7e59eedebd7a97c423986ea1c4c05/openhands-tools/openhands/tools/terminal/definition.py#L56-L102); [`tools/gemini/`](https://github.com/OpenHands/software-agent-sdk/blob/98de8bf3aca7e59eedebd7a97c423986ea1c4c05/openhands-tools/openhands/tools/gemini) |
| Industry | **Aider** (Python, ~50k stars) | **No tool registry, by design.** The model doesn't call tools to edit; it writes edits in a **text format** (whole files, search/replace blocks, unified diffs), which Aider parses and applies. The "registry" is a list of these *edit formats*, picked by name; an unknown one is an error listing the valid formats. Function-calling versions exist in the code but are **commented out** of the active list. | [`coders/__init__.py:16-34`](https://github.com/Aider-AI/aider/blob/5dc9490bb35f9729ef2c95d00a19ccd30c26339c/aider/coders/__init__.py#L16-L34); [`base_coder.py:61, 190-191`](https://github.com/Aider-AI/aider/blob/5dc9490bb35f9729ef2c95d00a19ccd30c26339c/aider/coders/base_coder.py#L190-L191); [`editblock_coder.py:18-41, 439`](https://github.com/Aider-AI/aider/blob/5dc9490bb35f9729ef2c95d00a19ccd30c26339c/aider/coders/editblock_coder.py#L18-L41); [`wholefile_func_coder.py:8-25`](https://github.com/Aider-AI/aider/blob/5dc9490bb35f9729ef2c95d00a19ccd30c26339c/aider/coders/wholefile_func_coder.py#L8-L25) |

**What the six agents with a tool registry agree on, and so what this plan follows:**
- **A lookup from tool name to tool**, and the loop finds tools by name (1b, 1e). Every agent with a registry does this, from 150-line tutorials to Gemini CLI, Codex, opencode and OpenHands.
- **An unknown tool is an error the model sees**, not a crash (1e). The better ones tell the model what *is* available (OpenHands lists the tools, opencode points to the tool list, Gemini CLI suggests the closest names). This also matches Phase 0's own rule: `read_file` returns errors instead of raising them.

**Where they differ:**
- **Where descriptions come from.** All six agents with a registry hand-write their descriptions. Only ghuntley and OpenHands generate anything automatically, and only the argument schema, from a typed class. Two industry agents go further the other way: Gemini CLI and OpenHands keep **different descriptions or tool sets for different models**, treating descriptions as prompt text to tune, not just documentation. This bears on decision 1 below, and on Phase 2's "same harness, different models" ([`ROADMAP.md:90`](../ROADMAP.md)).
- **Industry tool lists aren't fixed.** In Gemini CLI, Codex and opencode, which tools the model sees depends on config, policy, permissions or mode, and tools can arrive at runtime (discovery, MCP, external tools). The small projects have a fixed list. We'll meet the same pressures later: the approval prompt is step 5, MCP is step 14 ([`ROADMAP.md:100`](../ROADMAP.md)).
- **Bad arguments.** OpenHands validates arguments against the tool's schema and returns the error to the model, and tries to repair some malformed calls. This bears on decision 3 below.
- **Unknown-tool errors get smarter at scale.** The small projects just say "unknown"; the industry agents list or suggest alternatives, and Codex records the failure in telemetry, which is the kind of record Phase 2 (measurement) builds.
- **Text protocol vs native tool calling.** All the registry agents use **native tool calling** (the API receives the tool list and returns structured requests). We keep our text protocol for now; native tool calling is step 12, and comparing the two is step 13 ([`ROADMAP.md:98-99`](../ROADMAP.md)).
- **No registry at all** is a real design too, in two very different ways: mini-swe-agent has one shell tool (the roadmap's "Bash-only agent", Later, [`ROADMAP.md:187`](../ROADMAP.md)), and **Aider**, a major production agent, edits through text formats instead of tool calls, with its function-calling coders commented out. Text protocols like ours aren't just a teaching shortcut.

Not covered: Claude Code and Cursor aren't open source, so they'd appear only through their public docs; not read for this step.

### Sub-steps

From the problem, the evidence and the decisions below. Each one is a separate turn: built, run and discussed before the next.

| # | Sub-step | Replaces | Based on |
|---|---|---|---|
| 1a | **Look at the problem.** Print Phase 0's parser and `ask()` (its prompt is already printed in the setup above), and find the three hard-coded places. Also see what happens today to a malformed tool call. | — | The problem table above; decision 3 |
| 1b | **The registry.** One place that lists the tools: a dictionary from tool name to function. Register Phase 0's `read_file`, unchanged. Read what the registry can tell us about a tool: its parameters (from the signature) and its description (the docstring). | — | All six agents with a registry keep a lookup from name to tool; decision 1 |
| 1c | **Describe tools from the registry.** Build the prompt's tool list from what's registered: parameters from each signature, description from each docstring. Explain the format with a generic pattern **plus one concrete example line, generated from the first registered tool.** | `loop.py:10-14` | Gemini CLI builds the model's tool list from the registry; decisions 1 and 2 |
| 1d | **Parse any tool name, with three outcomes.** Phase 0's parser with two changes: (1) the prefix becomes `"tool: "` and the parser reads the tool's name, so any tool works; (2) where Phase 0 returned `None` on bad JSON, it raises a format error, plus one new check for arguments that aren't a JSON object. Outcomes: no tool call (a final answer), a valid call, or a malformed call (a format error). | `loop.py:22-29` | mini-swe-agent sends format errors back to the model; decision 3 |
| 1e | **Run a tool by name.** Look it up and call it. An unknown tool, or wrong argument names, come back as an error result listing what's available or expected (the expected parameters come from the signature). Try both failure cases. | `loop.py:51-52` | All six agents with a registry return unknown tools as errors; OpenHands lists available tools and returns argument errors; decisions 1 and 3 |
| 1f | **The new loop.** Phase 0's `ask()`, rewritten to use 1c–1e. A format error goes back to the model as a message, instead of ending the conversation. | `ask()` | learn-claude-code: *"the loop stays the same; only tool registration and dispatch grow"*; decision 3 |
| 1g | **Try it.** Phase 0's demo question, then print the conversation. Then register a tiny throwaway tool and check that it appears in the prompt and can be called, with no other change. | — | "Done when" checks 1 and 2 |

Not in this step, though industry agents do them: tools filtered by config or mode, tools added at runtime, "did you mean" suggestions, telemetry, per-model descriptions, and full type checking of arguments. These belong to later steps (approval, MCP, measurement, native tool calling).

### Decisions (agreed 2026-10-31)

1. **Where does a tool's description come from? Parameters from the function's signature; description from its docstring.** A description has two parts with different risks. Parameter names written separately can drift from the function, and then every call fails; generated from the signature, they can't. The description stays hand-written, as in all six agents with a registry, but it lives in the function itself, so each tool is described in one place. This is the hybrid ghuntley and OpenHands use (schema generated, description hand-written). It also prepares step 12: native tool calling needs a typed schema, which our type hints (`filename: str`) can provide. Not done: per-model descriptions like Gemini CLI's; with one model there's nothing to tune yet.
2. **Concrete example or generic pattern? Both:** a generic pattern explains the format, plus one real example line generated from the first registered tool, so nothing is hard-coded. A small model likely copies a concrete line more reliably; our only evidence so far is 1 bad run in 4 with a generic pattern alone, in code since removed. To watch: with several tools, the model may favor the one in the example. **This is measurable**, so it's a candidate Phase 2 experiment: example vs no example, on the same tasks.
3. **Arguments that aren't a JSON object? The parser tells three outcomes apart, and a malformed call goes back to the model as a format error.** Looking into this found a bigger problem in Phase 0: on bad JSON its parser returns `None` ([`loop.py:26-29`](../phase-0-foundation/agent_v0/loop.py)), which `ask()` treats as "no tool requested", so it **prints the broken call as the final answer and stops** ([`loop.py:46-49`](../phase-0-foundation/agent_v0/loop.py)). The model never learns it made a mistake. Now: the **parser** handles format (no call / valid call / malformed call), and **`run_tool`** handles content (unknown tool, wrong argument names). Not done: a validation library such as Pydantic; real type checking belongs to step 12, with real schemas.

### Done when

- With only `read_file` registered, the agent answers Phase 0's question the same way as before.
- Adding a tool needs only registering a function: no change to the prompt, parser or loop.
- Asking for an unknown tool, or passing wrong arguments, gives the model an error result instead of crashing.
- A malformed tool call goes back to the model as a format error, instead of ending the conversation.


### 1a. Look at the problem

Before changing anything, read Phase 0's code. `inspect.getsource` prints the source of an imported function, so we can read `agent_v0` without leaving the notebook. Look for every place that names `read_file`.

In [4]:
import inspect

print(inspect.getsource(parse_tool_call))
print(inspect.getsource(ask))

def parse_tool_call(reply: str):
    """Return the tool's arguments if the reply asks for read_file, else None."""
    prefix = "tool: read_file("
    for line in reply.splitlines():
        line = line.strip()
        if line.startswith(prefix) and line.endswith(")"):
            try:
                return json.loads(line[len(prefix):-1])
            except json.JSONDecodeError:
                return None
    return None

def ask(user_message: str, conversation: list, max_steps: int = 5) -> str:
    """Run the agent loop on one user message. Appends to `conversation`."""
    conversation.append({"role": "user", "content": user_message})

    for step in range(max_steps):
        reply = call_llm(conversation)                          # 1. ask the model
        conversation.append({"role": "assistant", "content": reply})

        args = parse_tool_call(reply)                           # 2. did it ask for a tool?
        if args is None:
            print("Assistant:", reply)         

Besides the system prompt printed in the setup, two places hard-code `read_file`:

- **The parser** only recognizes one tool: `prefix = "tool: read_file("`.
- **The loop** always calls one function: `result = read_file(args.get("filename", ""))`.

Now the hidden problem (decision 3). The parser returns `None` for two very different replies: a normal answer, and a tool call it couldn't read. Watch:

In [5]:
normal_answer = "It will print hello world."
broken_call = 'tool: read_file({filename: "sandbox/hello.py"})'   # missing quotes: not valid JSON

print(parse_tool_call(normal_answer))
print(parse_tool_call(broken_call))

None
None


Both return `None`. And in `ask()`, `None` means "no tool requested, this is the final answer". So a broken tool call is **printed as the agent's answer and the conversation stops**. The model never finds out it made a mistake.

**What 1a found:** three places to replace (prompt, parser, loop), and one silent failure to fix. Steps 1b–1f handle these.

### 1b. The registry

One place that lists the tools: a dictionary from **tool name** to **the Python function**. Registering a tool just adds it to the dictionary, under the function's own name.

We register Phase 0's `read_file` exactly as it is: nothing about the function changes.

In [6]:
TOOLS = {}


def register(function):
    """Make a Python function available to the model as a tool."""
    TOOLS[function.__name__] = function


register(read_file)
TOOLS

{'read_file': <function agent_v0.tools.read_file(filename: str) -> dict>}

Decision 1: everything the model needs to know about a tool comes from the function itself. Python can read it for us:

- the **parameters**, from the function's signature (`inspect.signature`);
- the **description**, from its docstring (the `"""..."""` text under `def`).

In [7]:
print(inspect.signature(read_file))                 # the full signature
print(list(inspect.signature(read_file).parameters))  # just the parameter names
print(read_file.__doc__)                             # the docstring

(filename: str) -> dict
['filename']
Read a text file and return its contents.


In [8]:
TOOLS

{'read_file': <function agent_v0.tools.read_file(filename: str) -> dict>}

### 1c. Describe tools from the registry

Now build the prompt from the registry instead of typing it. Three small pieces:

1. **One line per tool**: name, parameters and description.
2. **One concrete example line** (decision 2), generated from a registered tool, so it's real but not hard-coded.
3. **The prompt**, which puts them together with the same protocol as Phase 0.

In [9]:
def describe_tool(function) -> str:
    """One line for the prompt: name(parameters) - description."""
    params = ", ".join(inspect.signature(function).parameters)
    return f"  {function.__name__}({params}) - {function.__doc__}"


print(describe_tool(read_file))

  read_file(filename) - Read a text file and return its contents.


In [10]:
import json


def example_call(function) -> str:
    """A real example of the tool-call line, with placeholder values."""
    args = {name: "..." for name in inspect.signature(function).parameters}
    return f"tool: {function.__name__}({json.dumps(args)})"


print(example_call(read_file))

tool: read_file({"filename": "..."})


In [11]:
def build_system_prompt(tools: dict) -> str:
    tool_lines = "\n".join(describe_tool(f) for f in tools.values())
    first_tool = next(iter(tools.values()))
    return f"""You are a helpful coding assistant.

You have these tools:
{tool_lines}

To use a tool, reply with exactly one line and nothing else:
tool: tool_name({{"argument_name": "value"}})

For example:
{example_call(first_tool)}

After you receive a tool_result(...) message, use it to continue the task.
If you don't need a tool, just answer normally."""


print(build_system_prompt(TOOLS))

You are a helpful coding assistant.

You have these tools:
  read_file(filename) - Read a text file and return its contents.

To use a tool, reply with exactly one line and nothing else:
tool: tool_name({"argument_name": "value"})

For example:
tool: read_file({"filename": "..."})

After you receive a tool_result(...) message, use it to continue the task.
If you don't need a tool, just answer normally.


Compare with Phase 0's prompt in the setup above:

| | Phase 0 | Now |
|---|---|---|
| Tool line | Typed by hand: `read_file(filename) - returns the contents of a text file.` | Generated from the function's name, signature and docstring |
| Format | One exact line for `read_file` | A generic pattern **plus** one example generated from a registered tool |
| Adding a tool | Edit the prompt | Register the function; its line appears automatically |

Nothing uses this prompt yet. The loop (1f) will.

### 1d. Parse any tool name, with three outcomes

This is Phase 0's parser (printed in 1a) with **two changes**, and nothing else:

1. **Any tool name, not just `read_file`.** The prefix shrinks from `"tool: read_file("` to `"tool: "`, and the parser reads the name that follows. It returns `(tool_name, args)` instead of just `args`.
2. **Errors instead of a silent `None`.** Where Phase 0 returned `None` on bad JSON, this raises a `ToolFormatError` with a reason. One new check also catches arguments that aren't a JSON object (the bare-string case).

So the three outcomes are: no tool line → `None` (a final answer); a valid line → `(tool_name, args)`; an unreadable line → `ToolFormatError`. Raising an error follows mini-swe-agent, which raises a `FormatError` for badly formatted replies. The loop (1f) will catch it and send the message back to the model.

In [12]:
import json


class ToolFormatError(Exception):
    """The reply tried to call a tool, but the arguments couldn't be read."""


def parse_tool_call(reply: str):
    """Return (tool_name, args), None if there's no tool call, or raise ToolFormatError."""
    prefix = "tool: "                                                   # changed: any tool
    for line in reply.splitlines():
        line = line.strip()
        if line.startswith(prefix) and line.endswith(")") and "(" in line:
            name, _, raw_args = line[len(prefix):-1].partition("(")      # changed: read the name
            try:
                args = json.loads(raw_args)
            except json.JSONDecodeError:
                raise ToolFormatError(f"The arguments are not valid JSON: {raw_args}")   # changed
            if not isinstance(args, dict):                                               # new
                raise ToolFormatError(f'The arguments must be a JSON object, like {{"filename": "a.py"}}. Got: {raw_args}')
            return name, args                                           # changed: name too
    return None

Try it on the replies that matter, including the broken call from 1a and the bare-string call seen in an earlier test run:

In [13]:
replies = [
    "It will print hello world.",                              # no tool call
    'tool: read_file({"filename": "sandbox/hello.py"})',        # valid
    'tool: list_files({"folder": "."})',                        # valid, any tool name
    'tool: read_file({filename: "sandbox/hello.py"})',          # broken JSON (from 1a)
    'tool: read_file("sandbox/hello.py")',                      # a bare string, not an object
]

for reply in replies:
    try:
        print("OK:   ", parse_tool_call(reply))
    except ToolFormatError as e:
        print("ERROR:", e)

OK:    None
OK:    ('read_file', {'filename': 'sandbox/hello.py'})
OK:    ('list_files', {'folder': '.'})
ERROR: The arguments are not valid JSON: {filename: "sandbox/hello.py"}
ERROR: The arguments must be a JSON object, like {"filename": "a.py"}. Got: "sandbox/hello.py"


What to notice:

- The broken call from 1a is now an **error with a reason**, not a silent `None`.
- The bare string `"sandbox/hello.py"` is caught, with a message showing the right shape.
- `list_files` is accepted, though it doesn't exist. The parser only checks the **format**; whether the tool exists is checked when running it (1e).

### 1e. Run a tool by name

Phase 0 ran its tool with one line (printed in 1a):

```python
result = read_file(args.get("filename", ""))
```

`run_tool(name, args)` replaces it: look the tool up in the registry and call it with the model's arguments. `**args` turns `{"filename": "a.py"}` into `filename="a.py"`. Two things can go wrong, and both come back as an `{"error": ...}` result the model can read, like `read_file`'s own errors:

1. **An unknown tool.** The error lists the tools that do exist, as OpenHands does.
2. **Wrong arguments.** `inspect.signature(function).bind(**args)` checks the arguments against the function's parameters *without running it*. The error shows the arguments given and the ones expected.

(Phase 0's `args.get("filename", "")` quietly hid wrong arguments: a call with `{"path": ...}` became `read_file("")` and a confusing "File not found".)

In [14]:
def run_tool(name: str, args: dict) -> dict:
    """Call a registered tool. Unknown tools and wrong arguments become error results."""
    if name not in TOOLS:
        return {"error": f"Unknown tool: {name}. Available tools: {', '.join(TOOLS)}"}

    function = TOOLS[name]
    try:
        inspect.signature(function).bind(**args)
    except TypeError:
        expected = ", ".join(inspect.signature(function).parameters)
        return {"error": f"Wrong arguments for {name}. Got: {', '.join(args) or 'none'}. Expected: {expected}"}

    return function(**args)

Try the working case and both failure cases. The file is Phase 0's `hello.py`, which already exists.

In [15]:
print(run_tool("read_file", {"filename": "../phase-0-foundation/sandbox/hello.py"}))   # works
print(run_tool("list_files", {"folder": "."}))                                       # unknown tool
print(run_tool("read_file", {"path": "a.py"}))                                       # wrong argument name
print(run_tool("read_file", {}))                                                     # missing argument

{'filename': '../phase-0-foundation/sandbox/hello.py', 'content': 'name = "world"\nprint(f"hello {name}")\n'}
{'error': 'Unknown tool: list_files. Available tools: read_file'}
{'error': 'Wrong arguments for read_file. Got: path. Expected: filename'}
{'error': 'Wrong arguments for read_file. Got: none. Expected: filename'}


What to notice:

- Every outcome is a dict the loop can send back to the model; nothing crashes.
- Each error tells the model what to do instead: which tools exist, or which parameters are expected.
- Not handled here, same as Phase 0: a tool that crashes *while running*. That's each tool's own job, as `read_file` does for missing files.

### 1f. The new loop

Phase 0's `new_conversation()` and `ask()` (printed in 1a), copied exactly, with only these changes (marked in the code):

1. **`new_conversation()`** uses the prompt generated from the registry (1c) instead of the hand-written `SYSTEM_PROMPT`.
2. **The parser call** is wrapped in `try`: a `ToolFormatError` (1d) goes back to the model as a `tool_result` with the error, and the loop continues, instead of the broken call ending the conversation.
3. **Which tool:** the parser now returns `(name, args)`, so the loop unpacks both.
4. **Running it:** `run_tool(name, args)` (1e) replaces `read_file(args.get("filename", ""))`.

Everything else, including the step limit and how results are sent back, is unchanged.

In [16]:
def new_conversation() -> list:
    """A fresh conversation: just the system prompt."""
    return [{"role": "system", "content": build_system_prompt(TOOLS)}]   # changed: generated prompt


def ask(user_message: str, conversation: list, max_steps: int = 5) -> str:
    """Run the agent loop on one user message. Appends to `conversation`."""
    conversation.append({"role": "user", "content": user_message})

    for step in range(max_steps):
        reply = call_llm(conversation)                          # 1. ask the model
        conversation.append({"role": "assistant", "content": reply})

        try:                                                    # changed: the parser can raise
            call = parse_tool_call(reply)                       # 2. did it ask for a tool?
        except ToolFormatError as e:                            # new: a broken call goes back to the model
            print(f"  [step {step + 1}] format error: {e}")
            conversation.append(
                {"role": "user", "content": f"tool_result({json.dumps({'error': str(e)})})"}
            )
            continue
        if call is None:
            print("Assistant:", reply)                          # 3a. no -> final answer
            return reply

        name, args = call                                       # changed: which tool
        print(f"  [step {step + 1}] model asked for {name}({args})")
        result = run_tool(name, args)                           # 3b. changed: any registered tool
        conversation.append(                                    # 4. send the result back
            {"role": "user", "content": f"tool_result({json.dumps(result)})"}
        )

    print("Stopped: too many tool calls.")

Nothing runs yet: `ask()` calls the model, so it's tried in 1g. Note that a format error counts toward `max_steps`, so a model that keeps sending broken calls still stops after 5 steps.

### 1g. Try it

Two checks from "done when":

1. With only `read_file` registered, the agent answers Phase 0's question the same way as before.
2. A new tool works after just registering it: no change to the prompt, parser or loop.

First, the same safe playground as Phase 0: a `sandbox/` folder in this phase's folder.

In [17]:
from pathlib import Path

sandbox = Path("sandbox")
sandbox.mkdir(exist_ok=True)
(sandbox / "hello.py").write_text('name = "world"\nprint(f"hello {name}")\n');

**Check 1:** Phase 0's demo question.

In [18]:
conversation = new_conversation()
ask("What will sandbox/hello.py print when I run it?", conversation);

  [step 1] model asked for read_file({'filename': 'sandbox/hello.py'})
Assistant: When you run sandbox/hello.py, it will print:

```
hello world
```


Look inside, as in Phase 0: every message, including exactly what the model wrote when it asked for a tool.

In [19]:
for i, m in enumerate(conversation):
    print(f"{i}  {m['role']:9}  {m['content'][:80]!r}")

0  system     'You are a helpful coding assistant.\n\nYou have these tools:\n  read_file(filename)'
1  user       'What will sandbox/hello.py print when I run it?'
2  assistant  'tool: read_file({"filename": "sandbox/hello.py"})'
3  user       'tool_result({"filename": "sandbox/hello.py", "content": "name = \\"world\\"\\nprint'
4  assistant  'When you run sandbox/hello.py, it will print:\n\n```\nhello world\n```'


**Check 2:** a throwaway tool. Write a function, register it, and nothing else.

In [20]:
def count_lines(filename: str) -> dict:
    """Count the lines in a text file."""
    path = Path(filename)
    if not path.exists():
        return {"error": f"File not found: {filename}"}
    return {"filename": filename, "lines": len(path.read_text().splitlines())}


register(count_lines)
print(build_system_prompt(TOOLS))

You are a helpful coding assistant.

You have these tools:
  read_file(filename) - Read a text file and return its contents.
  count_lines(filename) - Count the lines in a text file.

To use a tool, reply with exactly one line and nothing else:
tool: tool_name({"argument_name": "value"})

For example:
tool: read_file({"filename": "..."})

After you receive a tool_result(...) message, use it to continue the task.
If you don't need a tool, just answer normally.


Its line appeared in the prompt on its own. Now ask a question that needs it:

In [21]:
conversation = new_conversation()
ask("How many lines are in sandbox/hello.py?", conversation);

  [step 1] model asked for count_lines({'filename': 'sandbox/hello.py'})
Assistant: The file sandbox/hello.py has 2 lines.


It was throwaway, so remove it again; section 2 adds the real tools.

In [22]:
del TOOLS["count_lines"]
list(TOOLS)

['read_file']